## 1. Загрузка данных
Эта часть подключает библиотеку `pandas`, читает CSV-файл и показывает первые строки датасета.

In [3]:
# Импортируем библиотеку pandas для работы с таблицами.
import pandas as pd

# Загружаем CSV-файл с данными Киноафиши в датафрейм df.
df = pd.read_csv("kinoafisha_data.csv")

# Смотрим первые строки, чтобы проверить, что файл прочитан корректно.
df.head()


,Страна,Название фильма,Год,Жанр,Рейтинг,Индекс
0,NaN,Monsta X: Connect X in Cinemas,2025,музыка,9.0,5
1,Австралия,Апгрейд,2018,боевик,7.8,551
2,Австралия,Апгрейд,2018,триллер,7.8,551
3,Австралия,Апгрейд,2018,фантастика,7.8,551
4,Австралия,Безумный Макс: Дорога ярости,2015,боевик,8.0,250


## 2. Первичный осмотр структуры
Здесь проверяется структура таблицы: колонки, типы данных и количество непустых значений.

In [4]:
# Смотрим общую информацию о датасете:
# количество строк, названия колонок, типы данных и наличие пропусков.
df.info()

# Отдельно выводим список колонок, чтобы убедиться, что нужные поля есть в данных.
df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3075 entries, 0 to 3074
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Страна           3074 non-null   object 
 1   Название фильма  3075 non-null   object 
 2   Год              3075 non-null   int64  
 3   Жанр             3075 non-null   object 
 4   Рейтинг          3075 non-null   float64
 5   Индекс           3075 non-null   int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 144.3+ KB


Index(['Страна', 'Название фильма', 'Год', 'Жанр', 'Рейтинг', 'Индекс'], dtype='object')

## 3. Проверка пропусков
Проверяется, есть ли в столбцах значения `NaN`/`NULL`.

In [5]:
# Проверяем количество пропусков NaN/NULL в каждом столбце.
# Это базовая проверка полноты данных.
missing_values = df.isna().sum()
missing_values


,0
Страна,1
Название фильма,0
Год,0
Жанр,0
Рейтинг,0
Индекс,0


## 4. Проверка пустых строк
Отдельно ищутся пустые текстовые значения, которые не всегда считаются пропусками.

In [6]:
# Проверяем пустые строки, которые не считаются NaN,
# например "" или значения, состоящие только из пробелов.
empty_strings = (df.astype(str).apply(lambda col: col.str.strip()) == "").sum()
empty_strings


,0
Страна,0
Название фильма,0
Год,0
Жанр,0
Рейтинг,0
Индекс,0


## 5. Проверка полных дубликатов
Проверяется, повторяются ли строки полностью по всем столбцам.

In [7]:
# Ищем полные дубликаты строк.
# Полный дубликат — это строка, которая полностью повторяет другую строку по всем колонкам.
full_duplicates = df[df.duplicated()]
full_duplicates


,Страна,Название фильма,Год,Жанр,Рейтинг,Индекс


## 6. Проверка потенциальных дублей фильмов
Проверяются повторы по названию, году и стране. Для этого датасета такие повторы могут быть нормой, потому что фильм может иметь несколько жанров.

In [8]:
# Ищем потенциальные дубликаты фильмов по ключевым полям:
# название фильма + год + страна.
# Важно: в этом датасете один фильм может повторяться по разным жанрам,
# поэтому такие строки нужно рассматривать как потенциальные дубли, а не как автоматическую ошибку.
movie_duplicates = df[
    df.duplicated(
        subset=["Название фильма", "Год", "Страна"],
        keep=False
    )
]

movie_duplicates


,Страна,Название фильма,Год,Жанр,Рейтинг,Индекс
1,Австралия,Апгрейд,2018,боевик,7.8,551
2,Австралия,Апгрейд,2018,триллер,7.8,551
3,Австралия,Апгрейд,2018,фантастика,7.8,551
4,Австралия,Безумный Макс: Дорога ярости,2015,боевик,8.0,250
5,Австралия,Безумный Макс: Дорога ярости,2015,приключения,8.0,250
...,...,...,...,...,...,...
3070,Япония,Ходячий замок,2004,сказка,8.5,50
3071,Япония,Ходячий замок,2004,фантастика,8.5,50
3072,Япония,Человек-бензопила. Фильм: История Резе,2025,анимация,8.8,15
3073,Япония,Человек-бензопила. Фильм: История Резе,2025,боевик,8.8,15


## 7. Проверка рейтинга
Рейтинг должен быть числом в диапазоне от 0 до 10.

In [9]:
# Приводим рейтинг к числовому типу.
# Если значение нельзя преобразовать в число, оно станет NaN.
df["Рейтинг"] = pd.to_numeric(df["Рейтинг"], errors="coerce")

# Ищем некорректные рейтинги:
# пустые значения, значения меньше 0 или больше 10.
rating_issues = df[
    (df["Рейтинг"].isna()) |
    (df["Рейтинг"] < 0) |
    (df["Рейтинг"] > 10)
]

rating_issues


,Страна,Название фильма,Год,Жанр,Рейтинг,Индекс


## 8. Проверка года выпуска
Год должен быть числом в допустимом диапазоне.

In [10]:
# Приводим год к числовому типу.
# Некорректные значения будут заменены на NaN.
df["Год"] = pd.to_numeric(df["Год"], errors="coerce")

# Ищем некорректные годы:
# пустые значения, годы раньше появления кино или годы в будущем.
year_issues = df[
    (df["Год"].isna()) |
    (df["Год"] < 1888) |
    (df["Год"] > 2026)
]

year_issues


,Страна,Название фильма,Год,Жанр,Рейтинг,Индекс


## 9. Проверка жанров
Жанр должен быть заполнен, потому что он используется для группировки фильмов.

In [11]:
# Проверяем поле "Жанр" на пропуски и пустые строки.
# Жанр нужен для анализа фильмов по категориям.
genre_issues = df[
    df["Жанр"].isna() |
    (df["Жанр"].astype(str).str.strip() == "")
]

genre_issues


,Страна,Название фильма,Год,Жанр,Рейтинг,Индекс


## 10. Анализ категорий жанров
Показывает список жанров и частоту их использования.

In [12]:
# Считаем, сколько раз встречается каждый жанр.
# Это помогает увидеть справочник жанров и найти странные или редкие значения.
genre_categories = (
    df["Жанр"]
    .dropna()
    .astype(str)
    .str.strip()
    .value_counts()
)

genre_categories


,count
Жанр,
драма,611
приключения,284
боевик,275
комедия,250
триллер,243
анимация,184
фантастика,157
мелодрама,157
семейный,146


## 11. Анализ категорий стран
Показывает список стран и частоту их использования.

In [13]:
# Считаем, сколько раз встречается каждая страна.
# Это помогает проверить справочник стран и найти неоднородные значения.
country_categories = (
    df["Страна"]
    .dropna()
    .astype(str)
    .str.strip()
    .value_counts()
)

country_categories


,count
Страна,
США,1663
СССР,245
Великобритания,217
Япония,154
Франция,150
Россия,124
Германия,91
Испания,46
Южная Корея,45


## 12. Проверка лишних пробелов
Ищет значения с пробелами в начале или в конце текста.

In [14]:
# Проверяем наличие лишних пробелов в жанрах.
# Если значение отличается от значения после strip(), значит в нем есть пробелы по краям.
genre_spaces = df[
    df["Жанр"].astype(str) != df["Жанр"].astype(str).str.strip()
]

# Аналогично проверяем лишние пробелы в странах.
country_spaces = df[
    df["Страна"].astype(str) != df["Страна"].astype(str).str.strip()
]

genre_spaces, country_spaces


(Empty DataFrame
 Columns: [Страна, Название фильма, Год, Жанр, Рейтинг, Индекс]
 Index: [],
 Empty DataFrame
 Columns: [Страна, Название фильма, Год, Жанр, Рейтинг, Индекс]
 Index: [])

## 13. Проверка регистра категорий
Помогает найти случаи, когда одна категория записана в разном регистре.

In [15]:
# Приводим жанры к нижнему регистру и считаем частоты.
# Это помогает проверить, нет ли дублей категорий из-за разного регистра.
genre_case_check = (
    df["Жанр"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .value_counts()
)

genre_case_check


,count
Жанр,
драма,611
приключения,284
боевик,275
комедия,250
триллер,243
анимация,184
фантастика,157
мелодрама,157
семейный,146


## 14. Итог
Эта часть нужна для фиксации результатов проверок в реестре DQ-проблем.

In [ ]:
# Финальная ячейка оставлена для вывода итогов DQ-проверок.
# По результатам этого блокнота заполняется файл dq_issues_register.xlsx.
